# Qwen 与 Gemma

In [1]:
import torch
import torch.nn as nn

## 1. Qwen —— Tie Word Embeddings（权重绑定）

Tie Word Embeddings 的意思是：**输入端的词向量矩阵** 和 **输出端的语言模型头（LM Head）矩阵** 共享同一套参数。

在很多模型里，这两块参数是分开的：

- `Token Embedding`：把词 ID 映射成向量。
- `LM Head`：把隐藏状态映射回词表上的概率分布。

而在 Qwen、GPT-2 这类模型中，这两个矩阵会“绑在一起”，也就是共用同一份权重。可以简单理解为：

- 输入时，用它把词 ID 变成向量；
- 输出时，再用同一套权重把向量映射回词表概率。

### 意义

1. **减少参数量**：词表通常很大，尤其是几十万词表时，如果输入和输出各存一份参数，会占很多内存。
2. **让训练更一致**：输入端学到的词语表示，会直接影响输出端的预测，两个方向的信息能更好地共享。
3. **实现更简洁**：很多大模型都会采用这种做法，属于很常见的工程优化。

In [2]:
class QwenTieEmbeddings(nn.Module):
    def __init__(self, vocab_size: int, hidden_size: int):
        super().__init__()
        # 1. 定义标准的 Embedding 层
        self.embed_tokens = nn.Embedding(vocab_size, hidden_size)
        
        # 2. 定义最后的 LM Head 预测层，注意不要 bias
        self.lm_head = nn.Linear(hidden_size, vocab_size, bias=False)
        
        # ==========================================
        # TODO 2: 将 lm_head 的权重在内存级别绑定到 embed_tokens 上
        # 这一步不是复制，而是让两个模块共享同一块参数内存。
        # 提示: 在 PyTorch 中，可以直接赋值 nn.Parameter 或是底层 tensor
        # self.lm_head.weight = ???
        # ==========================================
        # ???
        self.lm_head.weight = self.embed_tokens.weight  # 共享权重
        
    def forward_embed(self, input_ids):
        return self.embed_tokens(input_ids)
        
    def forward_lm_head(self, hidden_states):
        return self.lm_head(hidden_states)

## 2. Gemma —— 带偏置的 RMSNorm

标准 RMSNorm 的写法通常是：

$$
\mathrm{RMSNorm}(x) = \gamma \cdot \frac{x}{\sqrt{\frac{1}{d} \sum_{i=1}^{d} x_i^2 + \varepsilon}}
$$

而 Gemma 的做法可以理解为：在缩放参数上再加一个 1，也就是让参数初始化后更接近“原样通过”的状态。

可以写成：

$$
\mathrm{GemmaRMSNorm}(x) = (1 + \gamma) \cdot \frac{x}{\sqrt{\frac{1}{d} \sum_{i=1}^{d} x_i^2 + \varepsilon}}
$$

### 意义

1. **训练初期更稳定**：如果 $\gamma$ 初始化得很小，那么一开始 $(1 + \gamma)$ 会非常接近 1，模型相当于先做一个“几乎不改变尺度”的归一化。
2. **梯度更平滑**：早期训练时不会因为缩放过大或过小而让输出分布突然变化。
3. **更符合残差网络的直觉**：很多大模型都希望一开始尽量保持“温和更新”，让网络先稳定起来，再逐步学出更复杂的变换。

In [3]:
class GemmaRMSNorm(nn.Module):
    def __init__(self, hidden_size: int, eps: float = 1e-6):
        super().__init__()
        self.eps = eps
        # weight 初始化为全 0
        self.weight = nn.Parameter(torch.zeros(hidden_size))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # 计算均方根
        x_f32 = x.float()
        variance = x_f32.pow(2).mean(-1, keepdim=True)
        x_norm = x_f32 * torch.rsqrt(variance + self.eps)
        
        # ==========================================
        # 先保留 FP32 的归一化结果，再做 +1 缩放。
        # TODO 1: 实现 Gemma 的 +1 缩放
        # 注意类型转换回 x.dtype
        # ==========================================
        # output = ???
        output = x_norm * (1 + self.weight)

        return output     

In [4]:
# 测试你的实现
def test_tricks():
    try:
        hidden_size = 64
        vocab_size = 1000
        
        # 1. 测试 Gemma RMSNorm
        gemma_norm = GemmaRMSNorm(hidden_size)
        x = torch.randn(2, 10, hidden_size)
        out = gemma_norm(x)
        
        # 验证初始化时 (weight=0)，输出等价于无缩放的 norm
        variance = x.float().pow(2).mean(-1, keepdim=True)
        expected = (x.float() * torch.rsqrt(variance + 1e-6)).to(x.dtype)
        
        assert torch.allclose(out, expected, atol=1e-4), "Gemma 的 1+w 缩放机制实现错误！"
        print("✅ Gemma RMSNorm (+1 trick) 测试通过！")
        
        # 2. 测试 Qwen 权重绑定
        qwen_model = QwenTieEmbeddings(vocab_size, hidden_size)
        
        # 检查物理内存地址是否相同
        ptr_embed = qwen_model.embed_tokens.weight.data_ptr()
        ptr_head = qwen_model.lm_head.weight.data_ptr()
        assert ptr_embed == ptr_head, "权重未在物理内存级别绑定！"
        
        # 模拟训练更新一次 Embedding
        qwen_model.embed_tokens.weight.data += 1.0
        
        # 验证 LM Head 的权重也跟着变了 (因为它们是同一个指针)
        assert qwen_model.lm_head.weight.data[0, 0] == qwen_model.embed_tokens.weight.data[0, 0], "权重更新未同步！"
        
        print("✅ Qwen Tie Word Embeddings 权重绑定测试通过！")
        print("\n架构变体技巧测试通过。")
        
    except NotImplementedError:
        print("请先完成 TODO 代码！")
        raise
    except (AttributeError, NameError, TypeError, ValueError) as e:
        print("代码未完成导致变量属性错误。" if isinstance(e, AttributeError) else "代码可能未完成，导致了类型错误")
        raise NotImplementedError("请先完成 TODO 代码！") from e
    except AssertionError as e:
        print(f"❌ 测试失败: {e}")
        raise NotImplementedError("请先完成 TODO 代码！") from e
    except Exception as e:
        print(f"❌ 发生未知异常: {e}")
        raise

test_tricks()

✅ Gemma RMSNorm (+1 trick) 测试通过！
✅ Qwen Tie Word Embeddings 权重绑定测试通过！

架构变体技巧测试通过。
